In [3]:
person_list = [(100, "Prashant", 30),
             (101, "David", None),
             (102, "Sushant",None),
             (103, "Abdul", 45),
             (104, "Shruti", 28)]

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Working with Null Values").getOrCreate()

person_df = spark.createDataFrame(person_list).toDF("id", "name", "age")

person_df.show()

+---+--------+----+
| id|    name| age|
+---+--------+----+
|100|Prashant|  30|
|101|   David|NULL|
|102| Sushant|NULL|
|103|   Abdul|  45|
|104|  Shruti|  28|
+---+--------+----+



2. How null equality is executed.

2.1 Find all records where age is 28.

2.1 Find all records where age is not given or unknown.

In [4]:
person_df.where("age==28").show()

+---+------+---+
| id|  name|age|
+---+------+---+
|104|Shruti| 28|
+---+------+---+



In [6]:
from pyspark.sql.functions import expr

person_df.withColumn("age_null_expr", expr("age IS null")).show()


+---+--------+----+-------------+
| id|    name| age|age_null_expr|
+---+--------+----+-------------+
|100|Prashant|  30|        false|
|101|   David|NULL|         true|
|102| Sushant|NULL|         true|
|103|   Abdul|  45|        false|
|104|  Shruti|  28|        false|
+---+--------+----+-------------+



In [7]:
person_df.where("age IS NULL").show()

+---+-------+----+
| id|   name| age|
+---+-------+----+
|101|  David|NULL|
|102|Sushant|NULL|
+---+-------+----+



3. How operators work on null values

3.1 Check the result of > operator on null values

In [9]:
person_df.withColumn("age_gt_29", expr("age > 29")).show()

+---+--------+----+---------+
| id|    name| age|age_gt_29|
+---+--------+----+---------+
|100|Prashant|  30|     true|
|101|   David|NULL|     NULL|
|102| Sushant|NULL|     NULL|
|103|   Abdul|  45|     true|
|104|  Shruti|  28|    false|
+---+--------+----+---------+



In [10]:
person_df.where("age > 29").show()

+---+--------+---+
| id|    name|age|
+---+--------+---+
|100|Prashant| 30|
|103|   Abdul| 45|
+---+--------+---+



3.3 How mathametical operators work on null.

Calculate experience for every employee using the following formula.

experience = age - 23

In [11]:
person_df.withColumn("experience", expr("age - 23")).show()

+---+--------+----+----------+
| id|    name| age|experience|
+---+--------+----+----------+
|100|Prashant|  30|         7|
|101|   David|NULL|      NULL|
|102| Sushant|NULL|      NULL|
|103|   Abdul|  45|        22|
|104|  Shruti|  28|         5|
+---+--------+----+----------+



3.4. Calculate the experience knowing that age could be null.

If age is null then assume 23 years for experience calculation.

In [15]:
from pyspark.sql.functions import expr, nvl
person_df.withColumn("experience", expr("nvl(age, 23) - 23")).show()

+---+--------+----+----------+
| id|    name| age|experience|
+---+--------+----+----------+
|100|Prashant|  30|         7|
|101|   David|NULL|         0|
|102| Sushant|NULL|         0|
|103|   Abdul|  45|        22|
|104|  Shruti|  28|         5|
+---+--------+----+----------+



4. How aggregates work on null values

4.1 What is the average age?

In [16]:
person_df.selectExpr("avg(age) as average_age").show()

+------------------+
|       average_age|
+------------------+
|34.333333333333336|
+------------------+



In [17]:
person_df.where("age is not null").selectExpr("avg(age) as average_age").show()

+------------------+
|       average_age|
+------------------+
|34.333333333333336|
+------------------+

